In [12]:
import networkx as nx
import numpy as np
import scipy.stats as stats
from collections import defaultdict

G = nx.DiGraph()
nodes = [i for i in range(6)]
edges = [(0,1),(0,2),(1,3),(1,4),(2,3),(2,4),(3,5),(4,5),]
G.add_nodes_from(nodes)
G.add_edges_from(edges)
for u, v in G.edges():
    G.edges[u, v]['p'] = np.random.uniform(0.99, 1)
G.edges(data=True)


OutEdgeDataView([(0, 1, {'p': 0.9978956631959843}), (0, 2, {'p': 0.9946726982488239}), (1, 3, {'p': 0.9938638501417763}), (1, 4, {'p': 0.9909403736130274}), (2, 3, {'p': 0.9973522633106022}), (2, 4, {'p': 0.9954156704783953}), (3, 5, {'p': 0.9992197080865602}), (4, 5, {'p': 0.9900985847878363})])

# Importance Sampling

In a typical MC, $X_e \sim Bern(p_e)$, however, importance sampling utilizes $X_e \sim Bern(q_e): q_e < p_e$

Then, $w(X) = \prod_e \frac{p_e^{X_e}(1-p_e)^{1-X_e}}{q_e^{X_e}(1-q_e)^{1-X_e}}$

In [98]:
def sample_graph(G, q_scale=0.7):
    H = nx.DiGraph()
    H.add_nodes_from(G.nodes)

    log_w = 0.0
    for u, v, data in G.edges(data=True):
        p = data['p']
        
        # biased probability (push toward failure)
        q = p * q_scale
        q = min(q, 0.999)

        x = np.random.rand() < q

        # log weight correction
        if x:
            log_w += np.log(p / q)
        else:
            log_w += np.log((1 - p) / (1 - q))

        if x:
            H.add_edge(u, v)

    return H, np.exp(log_w)

def indicator(H):
    return nx.has_path(H, 0, 5)

def estimate_connectivity(N=10000, q_scale=0.9):
    """ most sample dominated"""
    estimates = []
    num, den = 0, 0
    for _ in range(N):
        H, w = sample_graph(G, q_scale=q_scale)

        I = 1 if indicator(H) else 0
        num += w*I
        den += w
        estimates.append(num/den)

    return np.mean(estimates), estimates

def estimate_failure(N=10000, q_scale=0.9):
    """ Rare sample dominated"""
    estimates = []

    for _ in range(N):
        H, w = sample_graph(G, q_scale=q_scale)

        I_fail = 1 - nx.has_path(H, 0, 5)
        estimates.append(I_fail * w)

    return np.mean(estimates), estimates

scale = 0.5
m1, est1 = estimate_connectivity(q_scale=scale)
m2, est2 = estimate_failure(q_scale=scale)
print(m1, 1-m2)

0.9998770107649507 0.9999783751806999
